# Paper figures

Single-pass preprocessing + plotting for every panel in `paper_figs.xlsx`.
Each section runs its own getters (cached, so re-runs are cheap), invokes
the relevant plots from `plotting.py`, then calls `pf.distribute('<folder>')`
to copy the produced files into `results/<folder>/<folder>_<panel>...<ext>`.
Plot calls that span multiple figures live in the earliest section per
xlsx order; later sections re-run them too so each section is self-contained.
Final cell runs `pf.parity_check()`.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent if Path.cwd().name == 'make_figures' else Path.cwd().resolve()))

import importlib
import pandas as pd

from utils import config
from utils.palettes import load_colors
from utils import cb_data, core_data, ol_data, ol_rf, polarity, vic
from utils import plotting as plot
plot = importlib.reload(plot)
from make_figures import paper_figs as pf

plot.set_html_output(True)

## Main Fig. 1


In [ ]:
# Preprocessing
inv         = core_data.get_inventory_meta()
meta        = core_data.get_meta()
ol_meta     = core_data.get_ol_meta()
ol_flow     = core_data.get_ol_flow_type()
flow_df     = core_data.get_flow().merge(meta[['idx','main_groups','sign']], on='idx', how='right')
flow_ol     = core_data.get_flow_ol()
conn_ol     = ol_data.get_ol_connectivity()
dir_ol      = ol_data.get_ol_directedness()
type_dir_ol = ol_data.get_ol_type_directedness()

colored_region, colored_main_groups, colored_sign, colored_seed = load_colors()
colors = {
    'region':      colored_region['color'].to_dict(),
    'main_groups': colored_main_groups['color'].to_dict(),
    'sign':        colored_sign['color'].to_dict(),
}

# Plots producing Main Fig. 1 panels
plot.plot_inventory_full_brain(inv, colors)                                     # 1d
plot.plot_inventory_right_ol(inv, colors)                                       # 1e

# Fig 1d / 1e inventory totals
_inv_all      = inv[inv.region != "other"]
_inv_all_sign = _inv_all[~_inv_all["sign"].isna()]
_inv_ol       = inv[(inv.region == "right OL") & (inv.main_groups != "nonOL")]
_inv_ol_sign  = _inv_ol[~_inv_ol["sign"].isna()]
print(f"Fig 1d panel 1 (neurons by region):      {_inv_all['bodyId'].nunique():,} neurons")
print(f"Fig 1d panel 2 (instances by region):    {_inv_all['cell_type'].nunique():,} instances")
print(f"Fig 1d panel 3 (neurons by sign):        {_inv_all_sign['bodyId'].nunique():,} neurons")
print(f"Fig 1d panel 4 (instances by sign):      {_inv_all_sign['cell_type'].nunique():,} instances")
print(f"Fig 1e panel 1 (neurons by main_groups): {_inv_ol['bodyId'].nunique():,} neurons")
print(f"Fig 1e panel 2 (instances by main_groups): {_inv_ol['cell_type'].nunique():,} instances")
print(f"Fig 1e panel 3 (neurons by sign):          {_inv_ol_sign['bodyId'].nunique():,} neurons")
print(f"Fig 1e panel 4 (instances by sign):        {_inv_ol_sign['cell_type'].nunique():,} instances")

plot.plot_coords(ol_meta)                                                  # 1b right
plot.plot_flow_by_main_groups(ol_flow, conn_ol, type_dir_ol, flow_ol, colored_main_groups)  # 1h_1, 1i_1

# Connection-level ff/la/fb breakdown for Fig 1i_1 (legacy nb 2 cells 5/6/7)
_thr = 0.5
_ff = conn_ol['hitting_time_diff'] >  _thr
_la = conn_ol['hitting_time_diff'].abs() <= _thr
_fb = conn_ol['hitting_time_diff'] < -_thr
_W = conn_ol.weight.sum(); _N = len(conn_ol)
print(f'Total weight fractions:      ff {conn_ol[_ff].weight.sum()/_W:.3f}, la {conn_ol[_la].weight.sum()/_W:.3f}, fb {conn_ol[_fb].weight.sum()/_W:.3f}')
print(f'Connection-count fractions:  ff {_ff.sum()/_N:.3f}, la {_la.sum()/_N:.3f}, fb {_fb.sum()/_N:.3f}')
print(f'E/I fractions:               ff is E {conn_ol[_ff & (conn_ol.sign_pre == 1)].weight.sum()/conn_ol[_ff].weight.sum():.3f}, '
      f'la is E {conn_ol[_la & (conn_ol.sign_pre == 1)].weight.sum()/conn_ol[_la].weight.sum():.3f}, '
      f'fb is I {conn_ol[_fb & (conn_ol.sign_pre == -1)].weight.sum()/conn_ol[_fb].weight.sum():.3f}')
plot.plot_flow_by_sign(ol_flow, conn_ol, type_dir_ol, flow_ol, colored_sign)               # 1h_2, 1i_2
plot.plot_flow_type_boxes(flow_df, dir_ol, colored_main_groups)                 # 1g left, 1j left, 1k_1
plot.plot_flow_type_scatter_and_triangle(ol_flow, type_dir_ol, colored_main_groups, colored_sign)  # 1k_2

# Mi4 / T4a pathway examples (1g right)
type_to_sign = dict(zip(ol_meta.cell_type, ol_meta.sign))
sign_to_color = colored_sign['color'].to_dict()
type_to_mg = dict(zip(ol_meta.cell_type, colored_main_groups.loc[ol_meta.main_groups, 'color']))
examples_dir = config.FIG_DIR / 'examples'
_layer_cumsum = {'T4a_R': 0.9, 'Mi4_R': 0.7}
for inst in core_data.load_layer_example_types():
    plot.plot_pathway(
        ol_data.get_paths_to_instance(inst), inst, ol_flow,
        thre_cumsum=_layer_cumsum[inst],
        neuron_to_color=type_to_mg,
        neuron_to_sign=type_to_sign, sign_color_map=sign_to_color,
        save_path=examples_dir / f'{config.FIG_DATASET}_{inst[:-2]}_pathway.pdf',
    )

pf.distribute('main_1')


## SI Fig. 1


In [ ]:
# Preprocessing (cached, no recompute if Main 1 ran)
ol_meta     = core_data.get_ol_meta()
ol_flow     = core_data.get_ol_flow_type()
meta        = core_data.get_meta()
flow_df     = core_data.get_flow().merge(meta[['idx','main_groups','sign']], on='idx', how='right')
flow_ol     = core_data.get_flow_ol()
conn_ol     = ol_data.get_ol_connectivity()
dir_ol      = ol_data.get_ol_directedness()
type_dir_ol = ol_data.get_ol_type_directedness()
layers      = ol_data.get_ol_layers()
roi_adj     = ol_data.get_ol_roi_adjacency()

colored_region, colored_main_groups, colored_sign, _ = load_colors()

# SI 1a/1b: fractional hit-hist groups & signs (additional outputs of by_main_groups / by_sign)
plot.plot_flow_by_main_groups(ol_flow, conn_ol, type_dir_ol, flow_ol, colored_main_groups)  # also frac
plot.plot_flow_by_sign(ol_flow, conn_ol, type_dir_ol, flow_ol, colored_sign)               # also frac, fb/la/ff signs

# Per-main-group connection-level ff/la/fb breakdown for SI 1b_1 / SI 1b_2
_thr = 0.5
print('SI 1b_1 / SI 1b_2 (by main_groups_pre):')
for _g in conn_ol['main_groups_pre'].dropna().unique():
    _sub = conn_ol[conn_ol['main_groups_pre'] == _g]
    _ff = _sub['hitting_time_diff'] >  _thr
    _la = _sub['hitting_time_diff'].abs() <= _thr
    _fb = _sub['hitting_time_diff'] < -_thr
    _W, _N = _sub.weight.sum(), len(_sub)
    _wE = lambda m, s: _sub[m & (_sub.sign_pre == s)].weight.sum() / _sub[m].weight.sum() if _sub[m].weight.sum() else float('nan')
    print(f'  {_g}:')
    print(f'    Total weight fractions:      ff {_sub[_ff].weight.sum()/_W:.3f}, la {_sub[_la].weight.sum()/_W:.3f}, fb {_sub[_fb].weight.sum()/_W:.3f}')
    print(f'    Connection-count fractions:  ff {_ff.sum()/_N:.3f}, la {_la.sum()/_N:.3f}, fb {_fb.sum()/_N:.3f}')
    print(f'    E/I fractions:               ff is E {_wE(_ff, 1):.3f}, la is E {_wE(_la, 1):.3f}, fb is I {_wE(_fb, -1):.3f}')
# SI 1c, 1d: ff/fb/la frac hist
plot.plot_flow_type_boxes(flow_df, dir_ol, colored_main_groups)                 # also fb_frac_hist_all_groups
# SI 1e: examples_0.1thre_no_vs_layer (extra output of plot_flow_type_scatter_and_triangle)
plot.plot_flow_type_scatter_and_triangle(ol_flow, type_dir_ol, colored_main_groups, colored_sign)
# SI 1f, 1g: connectivity heatmaps (visual input / OL internal / OL output / CB input + signed)
plot.plot_flow_connectivity_heatmap(conn_ol, colored_main_groups, colored_sign)
# SI 1h: threshold-fraction
plot.plot_flow_threshold_choice(conn_ol)
_y0 = conn_ol[conn_ol['hitting_time_diff'] > 0]['weight'].sum() / conn_ol['weight'].sum()
print(f'SI 1h: at post-pre layer threshold 0, fraction of weights above is {_y0:.3f}')
# SI 1i, 1j: per-layer violin (hitting_time) + per-layer hit-diff (post - pre)
side = config.SIDE_CHAR.upper()
me_rois = [f'ME_{side}_layer_{i:02d}' for i in range(1, 11)] + [f'AME({side})']
me_labels = [f'M{i}' for i in range(1, 11)] + ['AME']
lo_rois = (
    [f'LO_{side}_layer_{i}' for i in range(1, 8)]
    + [f'LOP_{side}_layer_{i}' for i in range(1, 5)]
)
lo_labels = ['LO1','LO2','LO3','LO4','LO5A','LO5B','LO6','LOP1','LOP2','LOP3','LOP4']
plot.plot_ol_layers_violin(layers, me_rois, me_labels, filename_stem='ME_layers_hit')
plot.plot_ol_layers_violin(layers, lo_rois, lo_labels, filename_stem='LOPLO_layers_hit')
plot.plot_ol_layers_hit_diff(roi_adj, me_rois, me_labels, filename_stem='ME_layers_hit')
plot.plot_ol_layers_hit_diff(roi_adj, lo_rois, lo_labels, filename_stem='LOPLO_layers_hit')

pf.distribute('si_1')


## SI Fig. 2


In [ ]:
ol_flow = core_data.get_ol_flow_type()
dir_ol  = ol_data.get_ol_directedness()
_, colored_main_groups, _, _ = load_colors()

# numerous1/numerous2 hit_hist + fb/la/ff (SI 2a, 2b)
plot.plot_flow_numerous_type_boxes(flow_ol, dir_ol, colored_main_groups)

pf.distribute('si_2')


## Main Fig. 2


In [ ]:
FRAC = 0.19

# Preprocessing
feat        = ol_data.get_ol_features()
clusters    = ol_data.get_ol_clusters(frac_thre=FRAC)
type_part   = ol_data.get_ol_type_participation(frac_thre=FRAC)
intra       = ol_data.get_ol_clusters_intra()
pathways    = ol_data.get_ol_connectivity_pathways(frac_thre=FRAC)
sankey      = ol_data.get_ol_sankey_connectivity(frac_thre=FRAC)
ol_meta     = core_data.get_ol_meta()
ol_flow     = core_data.get_ol_flow_type()
type_dir_ol = ol_data.get_ol_type_directedness()
cmp_ol      = polarity.get_ol_polarity_comparison()

n_clu = int(clusters.cluster.max())
cluster_num = clusters['cluster'].value_counts().sort_index()
in_instances = list(ol_meta[ol_meta.main_groups=='visual input'].instance.unique())

colored_region, colored_main_groups, colored_sign, colored_seed = load_colors()

# Clustering / pathway plots â€” produces 2d, 2e top, 2e bottom, 2g, 2h, 2i, 2j (+ SI 3a, 3b, 3c, 3e)
Z, thre, dendr = plot.plot_clusters_dendrograms(feat, frac_thre=FRAC)
plot.plot_clusters_correlation_matrix(feat, dendr, cluster_num)
plot.plot_clusters_example_features(feat, in_instances, [s+'_R' for s in plot._CLUSTER_TYPES], colored_seed)
# Cluster assignments for main_2_d example types
_ex_types_2d = [s + '_R' for s in plot._CLUSTER_TYPES]
_ex_cls_2d = clusters[clusters['instance'].isin(_ex_types_2d)][['instance', 'cluster']].sort_values('cluster')
print('main_2_d cluster assignments:')
print(_ex_cls_2d.to_string(index=False))
print('Largest cell type per OL cluster (by neuron count):')
_top_2d = clusters.loc[clusters.groupby('cluster')['count'].idxmax()][['cluster', 'instance', 'count']].sort_values('cluster')
print(_top_2d.to_string(index=False))
plot.plot_clusters_pathway_features(feat, in_instances, clusters, cluster_num, colored_seed)
plot.plot_clusters_scatter(clusters, ol_flow, type_dir_ol, colored_main_groups, colored_sign)
plot.plot_clusters_participation(type_part, ol_meta, ol_flow, colored_main_groups, colored_sign)
plot.plot_clusters_pathway_connectivity(pathways, n_clu)
plot.plot_clusters_sankey(sankey, colored_main_groups, colored_seed, colored_region)
plot.plot_clusters_graph(type_part, ol_meta, colored_main_groups, colored_sign, colored_seed)

# LC6 / LC4 / LPLC2 output pathway examples (2c right) and T4a participation (2f right)
type_to_sign  = dict(zip(ol_meta.cell_type, ol_meta.sign))
sign_to_color = colored_sign['color'].to_dict()
type_to_mg    = dict(zip(ol_meta.cell_type, colored_main_groups.loc[ol_meta.main_groups, 'color']))
examples_dir  = config.FIG_DIR / 'examples'
_out_cumsum = {'LC6_R': 0.4, 'LC4_R': 0.6, 'LPLC2_R': 0.6}
for inst, tc in _out_cumsum.items():
    plot.plot_pathway(
        ol_data.get_paths_to_instance(inst), inst, ol_flow,
        thre_cumsum=tc, neuron_to_color=type_to_mg,
        neuron_to_sign=type_to_sign, sign_color_map=sign_to_color,
        save_path=examples_dir / f'{config.FIG_DATASET}_{inst[:-2]}_pathway.pdf',
    )
_blues = ['#041f4a','#08306b','#08519c','#2171b5','#4292c6',
          '#6baed6','#9ecae1','#c6dbef','#deebf7']
type_to_color_part = type_to_mg | {
    s.split('_')[0]: _blues[c - 1]
    for s, c in zip(clusters['instance'], clusters['cluster'])
}
for inst in core_data.load_layer_example_types():
    plot.plot_pathway(
        ol_data.get_participation_paths(inst), inst, ol_flow,
        thre_cumsum=0.85, neuron_to_color=type_to_color_part,
        neuron_to_sign=type_to_sign, sign_color_map=sign_to_color,
        node_size=800, node_text_size=16,
        figsize=(9, 10), frac_text_pos=(0.2, 0.8), frac_text_size=16,
        save_path=examples_dir / f'{config.FIG_DATASET}_{inst[:-2]}_participation_pathway.pdf',
    )

# OL polarity bars (Main 2b)
cmp_vpn   = cmp_ol[cmp_ol['main_groups'] == 'OL output'].reset_index(drop=True)
cmp_other = cmp_ol[cmp_ol['main_groups'] != 'OL output'].reset_index(drop=True)
plot.plot_polarity_stacked_bars(cmp_vpn,   colored_seed, filename_stem='polarity_predictions_vpn')
_w_other = int(max(400, 35 * len(cmp_other) + 150) * 0.8 * 0.9)
plot.plot_polarity_stacked_bars(cmp_other, colored_seed, filename_stem='polarity_predictions_other', width=_w_other)

# Mismatched cell types per panel (Fig 2b_1 = other, Fig 2b_2 = vpn)
_miss_other = cmp_other[~cmp_other.match][['instance', 'polarity', 'predicted']]
_miss_vpn   = cmp_vpn  [~cmp_vpn.match  ][['instance', 'polarity', 'predicted']]
print(f'Fig 2b_1 (other OL): {cmp_other.match.mean():.1%} match ({len(cmp_other)} types); {len(_miss_other)} mismatches:')
print(_miss_other.to_string(index=False))
print(f'\nFig 2b_2 (VPN / OL output): {cmp_vpn.match.mean():.1%} match ({len(cmp_vpn)} types); {len(_miss_vpn)} mismatches:')
print(_miss_vpn.to_string(index=False))

pf.distribute('main_2')

In [ ]:
# --- OL neurons summary CSV (results/tables/) ---
_tables_dir = pf.RESULTS / 'tables'
_tables_dir.mkdir(parents=True, exist_ok=True)
_vic_ol_type = vic.get_ol_type_vic()
_type_ol_rf  = ol_rf.get_rf_type_ol()
_ol_csv = (
    ol_flow[['instance', 'count', 'sign', 'main_groups', 'hitting_time']]
    .copy()
    .rename(columns={'hitting_time': 'layer', 'main_groups': 'main_group'})
)
_ol_csv = _ol_csv.merge(
    type_dir_ol[['instance_pre', 'frac_ff', 'frac_la', 'frac_fb']].rename(
        columns={'instance_pre': 'instance', 'frac_la': 'frac_lat'}
    ),
    on='instance', how='left',
)
_ol_csv = _ol_csv.merge(_vic_ol_type, on='instance', how='left')
_ol_csv = _ol_csv.merge(clusters[['instance', 'cluster']], on='instance', how='left')
_ol_csv = _ol_csv.merge(
    _type_ol_rf.groupby('instance')[['size', 'r2']].median().reset_index(),
    on='instance', how='left',
)
_ol_csv = _ol_csv[['instance', 'count', 'sign', 'main_group', 'layer',
                    'frac_fb', 'frac_lat', 'frac_ff', 'VIC', 'cluster', 'size', 'r2']]
_ol_csv.to_csv(_tables_dir / f'{config.DATASET}_OL_neurons.csv', index=False)
print(f"Saved OL neurons CSV ({len(_ol_csv)} rows) -> results/tables/{config.DATASET}_OL_neurons.csv")

## SI Fig. 3


In [ ]:
FRAC = 0.19

feat        = ol_data.get_ol_features()
clusters    = ol_data.get_ol_clusters(frac_thre=FRAC)
ol_meta     = core_data.get_ol_meta()
intra       = ol_data.get_ol_clusters_intra()
intra_7     = ol_data.get_ol_clusters_intra(frac_thre=0.23)   # SI 3g
sweep       = ol_data.get_ol_lr_sweep()                       # SI 3f
match       = ol_data.get_ol_lr_cluster_match(frac_thre=FRAC) # SI 3j
_, colored_main_groups, _, colored_seed = load_colors()

n_clu = int(clusters.cluster.max())
cluster_num = clusters['cluster'].value_counts().sort_index()
in_instances = list(ol_meta[ol_meta.main_groups=='visual input'].instance.unique())

# SI 3a (full dendrogram), 3b (clustering matrix), 3c (outrel matrices)
Z, thre, dendr = plot.plot_clusters_dendrograms(feat, frac_thre=FRAC)
plot.plot_clusters_correlation_matrix(feat, dendr, cluster_num)
plot.plot_clusters_intra_connectivity(intra)             # SI 3c (n=13 outrel + wo_selfconn)
plot.plot_clusters_intra_connectivity(intra_7)           # SI 3g (outrel_7)

# SI 3d (no_thre clustering sweep), 3f (LR ARI), 3j (LR confusion)
n0 = int(ol_data.get_ol_clusters(frac_thre=FRAC, side_char='r')['cluster'].max())
plot.plot_lr_clustering_sweep(sweep, n0)                 # SI 3d, 3f
plot.plot_lr_cluster_confusion(match)                    # SI 3j
print(f'SI 3j OL L->R Hungarian-optimal cluster assignment (n_L={match["n_clu_L"]}, n_R={match["n_clu_R"]}):')
for r, c in zip(match['row_ind'], match['col_ind']):
    print(f'  L cluster {r + 1}  ->  R cluster {c + 1}')

# SI 3e: 13-cluster pathway features
FRAC_13 = 0.15
clusters_13 = ol_data.get_ol_clusters(frac_thre=FRAC_13)
cluster_num_13 = clusters_13['cluster'].value_counts().sort_index()
plot.plot_clusters_pathway_features(feat, in_instances, clusters_13, cluster_num_13, colored_seed, side_str='_13')

# SI 3h, 3i: left-side cluster pathway features
ol_meta_L = core_data.get_ol_meta(side_char='l')
feat_L    = ol_data.get_ol_features(side_char='l')
cl_L      = ol_data.get_ol_clusters(side_char='l', frac_thre=FRAC)
cluster_num_L = cl_L['cluster'].value_counts().sort_index()
in_L = list(ol_meta_L[ol_meta_L.main_groups=='visual input'].instance.unique())
plot.plot_clusters_example_features(feat_L, in_L, [s+'_L' for s in plot._CLUSTER_TYPES], colored_seed, side_str='_left')
plot.plot_clusters_pathway_features(feat_L, in_L, cl_L, cluster_num_L, colored_seed, side_str='_left')

pf.distribute('si_3')


## Main Fig. 3


In [ ]:
FRAC = 0.19

raw       = ol_rf.get_rf_raw_ol()
type_ol   = ol_rf.get_rf_type_ol()
edges_ol  = ol_rf.get_rf_connectivity_edges_ol(include_cb_post=False)
edges_cb  = ol_rf.get_rf_connectivity_edges_ol(include_cb_post=True)
ol_meta   = core_data.get_ol_meta()
stepsn    = ol_data.get_ol_stepsn_sum()
type_part = ol_data.get_ol_type_participation(frac_thre=FRAC)
clusters  = ol_data.get_ol_clusters(frac_thre=FRAC)
cmp_body  = ol_rf.get_rf_comparison_body()
cmp_type  = ol_rf.get_rf_comparison_type()
exp_sz    = ol_rf.get_experimental_rf_sizes()
exp_cc    = ol_rf.get_experimental_rf_sizes(sheet='RF size CC')
vic_type  = vic.get_ol_cb_vic_type().merge(
    core_data.get_flow().groupby('instance')['hitting_time'].median().reset_index(),
    on='instance', how='left',
)
rf_all   = ol_rf.get_rf_types_combined()
rf_vpn_vcbn = rf_all[(rf_all['main_groups'] == 'OL output') | (rf_all['region'] == 'CB')]

colored_region, colored_main_groups, colored_sign, _ = load_colors()
n_clu = int(clusters['cluster'].max())
cluster_names = [f'c{i}' for i in range(1, n_clu + 1)]
in_instances  = ['L1_R', 'L2_R', 'L3_R', 'R7_R', 'R8_R']
EXAMPLE_BID = 30134

# Main 3a: example RF positions/fits (3a_1..3a_4 from plot_rf_example; 3a_5 = LC6_R ellipses)
plot.plot_rf_example(ol_meta, stepsn, raw, EXAMPLE_BID, in_instances)
plot.plot_rf_example_pos_par(ol_meta, stepsn, raw, EXAMPLE_BID, in_instances)
# Main 3a_5 (LC6_R_rf_pos_par.pdf â€” fitted ellipses across all LC6 neurons) + Main 3b (examples_rf_*)
plot.plot_rf_positions_across_neurons(raw, 'LC6_R', example_bid=EXAMPLE_BID)
# Main 3c: direct-input RFs for LC6_R
roi_counts = pd.read_pickle(
    config.DATA_DIR / f'{config.DATASET}_OL_{config.SIDE_CHAR}_roi_counts.pkl'
)
input_raw = ol_rf.get_input_rf_raw_ol()
plot.plot_input_rf_example(roi_counts, input_raw, EXAMPLE_BID, 'LC6_R')
# Main 3d, 3e: experimental size comparisons
plot.plot_rf_size_example_comparison(cmp_body, star_instance='LC6_R', example_bid=EXAMPLE_BID)
plot.plot_rf_size_type_scatters(cmp_type, colored_main_groups)
plot.plot_rf_size_vs_experiment(cmp_type, exp_sz, colored_main_groups, colored_region)
plot.plot_rf_size_vs_experiment_cc(type_ol, exp_cc, colored_main_groups, colored_region)
# Main 3f: scatters + histograms by main groups / by sign
plot_cell_types = [s + '_R' for s in plot._FLOW_PLOT_TYPES]
plot.plot_rf_boxes_by_celltype(raw, plot_cell_types, colored_main_groups)
plot.plot_rf_scatters_by_main_groups(type_ol, colored_main_groups, colored_sign)
plot.plot_rf_scatters_by_cluster(
    type_ol, type_part, cluster_names=cluster_names,
    colored_main_groups_df=colored_main_groups,
)
plot.plot_rf_histograms(
    type_ol, type_part, cluster_names=cluster_names,
    colored_main_groups_df=colored_main_groups,
    colored_sign_df=colored_sign,
)
# Main 3g: hitting/no/amp vs size + cb_vic_vs_size
# (cb_vic_vs_size produced by plot_cb_rf_size_summary in CB-RF block)
plot.plot_cb_rf_size_summary(rf_vpn_vcbn, colored_region)
# Main 3h: sizediff internal + output
plot.plot_rf_sizediff_internal(edges_ol, colored_main_groups)
plot.plot_rf_sizediff_output(edges_cb, colored_region)
# Main 3i: connectivity boxes
plot.plot_rf_connectivity_boxes(edges_ol, colored_main_groups, colored_sign)

pf.distribute('main_3')

## Main Fig. 4


In [ ]:
FRAC_CB = 0.17
FRAC_OL = 0.19

feat_r       = cb_data.get_cb_features(side_char='r')
cl_r         = cb_data.get_cb_clusters(side_char='r', frac_thre=FRAC_CB)
conn         = cb_data.get_ol_cb_cluster_connectivity(ol_frac_thre=FRAC_OL, cb_frac_thre=FRAC_CB)
layer_lr     = cb_data.get_cb_layer_lr()
vic_type_cb  = vic.get_ol_cb_vic_type().merge(
    core_data.get_flow().groupby('instance')['hitting_time'].median().reset_index(),
    on='instance', how='left',
)
tbar_counts  = cb_data.get_cb_cluster_tbars_per_roi(side_char='r', cb_frac_thre=FRAC_CB)
type_part_cb_out = cb_data.get_ol_in_cb_type_participation(side_char='r', cb_frac_thre=FRAC_CB)
ol_clu       = ol_data.get_ol_clusters(side_char='r', frac_thre=FRAC_OL)
_meta        = core_data.get_meta()
flow_type_full = core_data.get_flow().groupby('instance')['hitting_time'].median().reset_index()

colored_region, colored_main_groups, colored_sign, colored_seed = load_colors()
cluster_num   = cl_r['cluster'].value_counts().sort_index()
in_instances  = [c for c in feat_r.columns if c.endswith('_R')]
cb_example_types = plot._CB_CLUSTER_TYPES[config.SIDE_CHAR]

VIC_THRE = 5e-4
_vic_raw = vic.get_ol_cb_vic_raw()
_vpn_above_raw  = _vic_raw[(_vic_raw.main_groups == 'OL output') & (_vic_raw.VIC > VIC_THRE)]
_cb_above_raw   = _vic_raw[(_vic_raw.region == 'CB') & (_vic_raw.VIC > VIC_THRE)]
_vpn_above_type = vic_type_cb[(vic_type_cb.region == 'right OL') & (vic_type_cb.VIC > VIC_THRE)]
_cb_above_type  = vic_type_cb[(vic_type_cb.region == 'CB') & (vic_type_cb.VIC > VIC_THRE)]
print(f"VPN above VIC threshold ({VIC_THRE:.0e}): {_vpn_above_raw['bodyId'].nunique():,} neurons, {_vpn_above_type['instance'].nunique():,} cell types")
print(f"CB  above VIC threshold ({VIC_THRE:.0e}): {_cb_above_raw['bodyId'].nunique():,} neurons, {_cb_above_type['instance'].nunique():,} cell types")

# Load light-sensitive cell types (no '?' in name, no 'N?' in Light sensitivity column)
_lit_df = pd.read_excel(
    config.PARAMS_DIR / 'fly_literature_experiments.xlsx',
    sheet_name='light sensitivity',
)
_light_types = []
for _, _row in _lit_df.iterrows():
    if str(_row['Light sensitivity']).strip() == 'N?':
        continue
    for _t in str(_row['Neuron type']).split(','):
        _t = _t.strip()
        if _t and '?' not in _t:
            _light_types.append(_t)
_light_types = list(dict.fromkeys(_light_types))  # deduplicate, preserve order
# Match to instances present in vic_type_cb (append side suffix)
_vic_instances = set(vic_type_cb['instance'].values)
_light_highlight = [
    t + f'_{config.SIDE_CHAR.upper()}'
    for t in _light_types
    if t + f'_{config.SIDE_CHAR.upper()}' in _vic_instances
]
print(f"Light-sensitive highlight types found in VIC data: {len(_light_highlight)}")

# Main 4a: VIC scatter / layer histogram / cumsum
# Main 4_a_1/2/3: VPNs colored by OL output blue (main_groups), CB stays dark gray (region).
_vic_colors = colored_region.copy()
_vic_colors.loc['right OL', 'color'] = colored_main_groups.loc['OL output', 'color']
plot.plot_vic_vs_layer_scatter(vic_type_cb, _vic_colors,
    highlight=_light_highlight,
    show_highlight_labels=False,
    matched_instances=cb_example_types + ['aMe12_R', 'VS_R'])
plot.plot_vic_layer_histogram(vic_type_cb, _vic_colors)
plot.plot_vic_cumsum(vic_type_cb, _vic_colors)
# Main 4c, 4d top: in-out examples + pathways for CB clusters
plot.plot_cb_clusters_example_features(feat_r, in_instances, cb_example_types, colored_seed)
# Cluster assignments for main_4_c example types
_ex_cls_4c = cl_r[cl_r['instance'].isin(cb_example_types)][['instance', 'cluster']].sort_values('cluster')
print('main_4_c cluster assignments:')
print(_ex_cls_4c.to_string(index=False))
print('Largest cell type per CB cluster (by neuron count):')
_top_4c = cl_r.loc[cl_r.groupby('cluster')['count'].idxmax()][['cluster', 'instance', 'count']].sort_values('cluster')
print(_top_4c.to_string(index=False))
plot.plot_cb_clusters_pathway_features(feat_r, in_instances, cl_r, cluster_num, colored_seed)
# Main 4d bottom + SI 4c: CB dendrograms
plot.plot_clusters_dendrograms(feat_r, frac_thre=FRAC_CB, side_str='_cb', save_dir=config.FIG_DIR / 'CB_pathways')
# Main 4e: CB cluster vs layer scatter
plot.plot_cb_clusters_scatter(cl_r, layer_lr, colored_region, highlight=cb_example_types)
# Main 4f: CB cluster graph
_fig4f_types = ["aMe4", "LPT111", "VS", "aMe12", "LC4", "LC6", "LPLC2", "T4a", "Tm5a"]
plot.plot_cb_clusters_graph(type_part_cb_out, ol_clu, colored_region, ol_highlight=_fig4f_types)
# Main 4g top: CB cluster x ROI heatmap
plot.plot_cb_clusters_rois(tbar_counts)

# pC1_2a / pC1_5b CB pathway examples (Main 4b right)
type_to_sign_full  = dict(zip(_meta.cell_type, _meta.sign))
sign_to_color_full = colored_sign['color'].to_dict()
_cb_color_df = pd.concat([
    pd.DataFrame(colored_region.loc['CB', 'color'], index=['nonOL'], columns=['color']),
    colored_main_groups,
])
type_to_color_cb = dict(zip(_meta.cell_type, _cb_color_df.loc[_meta.main_groups, 'color']))
examples_dir_cb = config.FIG_DIR / 'examples'
for inst, thre_cumsum, h_scale in [('pC1_2a_R', 0.6, 0.64), ('pC1_5b_R', 0.2, 0.65)]:
    paths = cb_data.get_cb_paths_to_instance(inst)
    hit_inst = paths['hit_inst']
    plot.plot_pathway(
        paths, inst, flow_type_full,
        thre_cumsum=thre_cumsum,
        figsize=(1.5 + hit_inst * 3, 8 * h_scale),
        neuron_to_color=type_to_color_cb,
        neuron_to_sign=type_to_sign_full, sign_color_map=sign_to_color_full,
        node_size=1400, node_text_size=18, highlight=False,
        save_path=examples_dir_cb / f'{config.FIG_DATASET}_{inst[:-2]}_pathway.pdf',
    )

pf.distribute('main_4')


In [ ]:
# --- VCBNs summary CSV (results/tables/) ---
_tables_dir = pf.RESULTS / 'tables'
_tables_dir.mkdir(parents=True, exist_ok=True)
_type_dir_ol_4 = ol_data.get_ol_type_directedness()
_rf_cb_type    = ol_rf.get_rf_type_cb()
_vcbn_meta     = _meta[core_data.is_cb_neuron(_meta)][['instance', 'sign']].drop_duplicates('instance')
_count_all     = core_data.get_flow()[['instance', 'count']].drop_duplicates('instance')
_vcbn_csv = (
    _vcbn_meta
    .merge(_count_all, on='instance', how='left')
    .merge(flow_type_full.rename(columns={'hitting_time': 'layer'}), on='instance', how='left')
    .merge(
        _type_dir_ol_4[['instance_pre', 'frac_ff', 'frac_la', 'frac_fb']].rename(
            columns={'instance_pre': 'instance', 'frac_la': 'frac_lat'}
        ),
        on='instance', how='left',
    )
)
_vic_vcbn = vic_type_cb[vic_type_cb['region'] == 'CB'][['instance', 'VIC']]
_vcbn_csv = _vcbn_csv.merge(_vic_vcbn, on='instance', how='left')
_vcbn_csv = _vcbn_csv.merge(cl_r[['instance', 'cluster']], on='instance', how='left')
_vcbn_csv = _vcbn_csv.merge(
    _rf_cb_type.groupby('instance')[['size', 'r2']].median().reset_index(),
    on='instance', how='left',
)
_vcbn_csv = _vcbn_csv[['instance', 'count', 'sign', 'layer',
                        'frac_fb', 'frac_lat', 'frac_ff', 'VIC', 'cluster', 'size', 'r2']]
_vcbn_csv.to_csv(_tables_dir / f'{config.DATASET}_VCBNs.csv', index=False)
print(f"Saved VCBNs CSV ({len(_vcbn_csv)} rows) -> results/tables/{config.DATASET}_VCBNs.csv")

## SI Fig. 4


In [ ]:
FRAC_CB = 0.17
FRAC_OL = 0.19

feat_r       = cb_data.get_cb_features(side_char='r')
cl_r         = cb_data.get_cb_clusters(side_char='r', frac_thre=FRAC_CB)
sweep        = cb_data.get_cb_lr_sweep()
match        = cb_data.get_cb_lr_cluster_match(frac_thre=FRAC_CB)  # threshold-based; left side has 11 clusters
layer_lr     = cb_data.get_cb_layer_lr()
layer_lr_hom = cb_data.get_cb_layer_lr_homologue()
vic_homologue  = vic.get_cb_vic_lr_homologue()
vic_binoc_type = vic.get_cb_vic_binocular_type()
intra_cb     = cb_data.get_cb_clusters_intra(frac_thre=FRAC_CB)
type_part_cb_out = cb_data.get_ol_in_cb_type_participation(side_char='r', cb_frac_thre=FRAC_CB)

colored_region, colored_main_groups, _, colored_seed = load_colors()
n0 = int(cl_r['cluster'].max())

# SI 4a: CB layer L/R comparison + horizontal left layer histogram
plot.plot_cb_layer_lr_comparison(layer_lr, colored_region, colored_main_groups)
plot.plot_cb_layer_hist_left(layer_lr_hom, colored_region, colored_main_groups)
# SI 4b: CB VIC L/R comparison
plot.plot_cb_vic_lr_homologue(vic_homologue, colored_region)
# SI 4c: full CB dendrogram
plot.plot_clusters_dendrograms(feat_r, frac_thre=FRAC_CB, side_str='_cb', save_dir=config.FIG_DIR / 'CB_pathways')
# SI 4d: CB intra-cluster connectivity (cb_outrel_conn_matrix_clusters)
plot.plot_cb_clusters_intra_connectivity(intra_cb)
# SI 4e, 4f: CB no_thre + LR_ARI sweep + L/R confusion
plot.plot_cb_lr_clustering_sweep(sweep, n0)
plot.plot_cb_lr_cluster_confusion(match)
print(f'SI 4i CB L->R Hungarian-optimal cluster assignment (n_L={match["n_clu_L"]}, n_R={match["n_clu_R"]}):')
for r, c in zip(match['row_ind'], match['col_ind']):
    print(f'  L cluster {r + 1}  ->  R cluster {c + 1}')
# SI 4g, 4h: left-side CB pathway examples + pathways + dendrogram (9 clusters to parallel right side)
feat_cb_L = cb_data.get_cb_features(side_char='l')
cl_cb_L   = cb_data.get_cb_clusters(side_char='l', frac_thre=FRAC_CB)
cluster_num_cb_L = cl_cb_L['cluster'].value_counts().sort_index()
in_cb_L = [c for c in feat_cb_L.columns if c.endswith('_L')]
plot.plot_cb_clusters_example_features(feat_cb_L, in_cb_L, plot._CB_CLUSTER_TYPES['l'], colored_seed, side_str='_left')
plot.plot_cb_clusters_pathway_features(feat_cb_L, in_cb_L, cl_cb_L, cluster_num_cb_L, colored_seed, side_str='_left')
# SI 4h bottom: left-CB dendrogram (parallels Main 4d bottom cb_dendrogram_clusters.pdf)
plot.plot_clusters_dendrograms(feat_cb_L, frac_thre=FRAC_CB, side_str='_left_cb', save_dir=config.FIG_DIR / 'CB_pathways', compact_lastp=11)
# SI 4j: CB cluster output / input graphs (uses ol_cb_cluster_connectivity)
conn = cb_data.get_ol_cb_cluster_connectivity(ol_frac_thre=FRAC_OL, cb_frac_thre=FRAC_CB)
plot.plot_ol_cb_cluster_connectivity(conn)
# SI 4k: CB participation n_clu out
plot.plot_cb_participation_n_clu_out(type_part_cb_out, colored_main_groups)
# SI 4l: binoc VIC L/R
plot.plot_cb_vic_binocularity(vic_binoc_type, colored_region)

pf.distribute('si_4')


## Main Fig. 5


In [ ]:
coverage = cb_data.get_ol_roi_coverage(seed=0)
sectors  = core_data.get_sector_map()

# Loops over all ROIs producing {full, relative, symbol}.pdf each.
# Main 5b left = ME(R)_{full,relative,symbol}; Main 5b right = symbol files of every other ROI.
plot.plot_ol_roi_coverage_per_roi(coverage, sectors)

pf.distribute('main_5')


## SI Fig. 7


In [ ]:
FRAC_CB = 0.19

rf_all   = ol_rf.get_rf_types_combined()
rf_vpn_vcbn = rf_all[(rf_all['main_groups'] == 'OL output') | (rf_all['region'] == 'CB')]
rf_type_cb = ol_rf.get_rf_type_cb()
cl_r       = cb_data.get_cb_clusters(side_char='r', frac_thre=FRAC_CB)
tb_clu     = cb_data.get_ol_to_cb_weights()
coverage   = cb_data.get_ol_roi_coverage(seed=0)
_sizes     = rf_all.set_index('instance')['size']

colored_region, colored_main_groups, _, _ = load_colors()
# SI 7a/7b: VPNs colored by OL output blue (main_groups), CB stays dark gray (region) â€” mirrors Main 4a.
_rf_colors = colored_region.copy()
_rf_colors.loc['right OL', 'color'] = colored_main_groups.loc['OL output', 'color']

# SI 7a: CB layer vs r2 + r2 cumsum
plot.plot_cb_rf_r2_summary(rf_vpn_vcbn, _rf_colors)
# SI 7b: CB layer vs size + size hist
plot.plot_cb_rf_size_summary(rf_vpn_vcbn, _rf_colors)
# SI 7c: examples size vs cluster
plot.plot_cb_clusters_size_vs_cluster(cl_r, rf_type_cb, colored_region)
# SI 7d: OL->CB highres subset
plot.plot_out_cb_highres_sub(tb_clu, _sizes)
# SI 7e: relative coverage per ROI summary
plot.plot_ol_roi_coverage_summary(coverage)

pf.distribute('si_7')


## Main Fig. 6


In [ ]:
rf_all = ol_rf.get_rf_types_combined()
plot.plot_rf_size_per_layer(rf_all)

pf.distribute('main_6')


## Parity check


In [ ]:
pf.parity_check()
